# Reviewer-auditable cross-lingual morphology pipeline

This notebook replaces the legacy stateful notebook. It runs the audited data preparation, controlled morphology views, immutable model pinning, the pre-specified 30-run GPU matrix, strict three-seed aggregation, hierarchical paired bootstrap inference, and blinded linguistic error analysis.

**Important:** no code can guarantee journal acceptance. This workflow is designed to produce auditable evidence and to fail rather than report incomplete or mutable results. Smoke tests, missing seeds, mutable model revisions, and unavailable Hungarian data are never promoted to paper results.

A language is an evaluation group, not a separate training job: each Turkish-trained model/seed is evaluated on every available test language. Therefore the primary block is 4 models × 3 seeds = 12 training runs, not 48 duplicated training runs. The full design contains 30 training runs.


## 1. Mount Drive and obtain the exact repository branch

Select **Runtime → Change runtime type → A100 GPU** before running. Results and caches are kept in Drive so interrupted sessions can resume.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, os, shlex, shutil, subprocess, sys

REPO_URL = "https://github.com/habibekarayigit/HBB_multilingual.git"
REPO_BRANCH = "arena/ca84efea-hbb-multilingual"
PROJECT_ROOT = Path("/content/drive/MyDrive/HBB_multilingual")


def run(command, *, cwd=None):
    command = [str(item) for item in command]
    print("$", shlex.join(command), flush=True)
    return subprocess.run(command, cwd=cwd or PROJECT_ROOT, check=True)


PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if not (PROJECT_ROOT / ".git").exists():
    run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, PROJECT_ROOT], cwd=PROJECT_ROOT.parent)
else:
    current_branch = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "branch", "--show-current"], text=True
    ).strip()
    if current_branch != REPO_BRANCH:
        raise RuntimeError(f"Expected {REPO_BRANCH}, found {current_branch}. Use a separate folder.")
    run(["git", "fetch", "origin", REPO_BRANCH])
    run(["git", "merge", "--ff-only", f"origin/{REPO_BRANCH}"])

os.chdir(PROJECT_ROOT)
print("Repository:", PROJECT_ROOT)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Configuration

Do not point `HU_FILE` to a substitute dataset. Leave it as `None` unless the exact licensed Hungarian file and its label semantics have been verified.


In [ ]:
# Exact Hungarian source only; otherwise keep None.
HU_FILE = None  # Example: Path("/content/drive/MyDrive/SIGTURK/hungarian_toxic.csv")
HU_TEXT_COLUMN = "text"
HU_LABEL_COLUMN = "toxic"
HU_THRESHOLD = 2.5

REPIN_MODEL_REVISIONS = False
REBUILD_MORPHOLOGY = False
MORPH_QC_APPROVED = False  # Set True only after the manual QC sheet has been completed and reviewed.
RUN_GPU_MATRIX = True
ERRORS_PER_LANGUAGE_OUTCOME = 100  # Up to 100 FP and 100 FN per language.

REQUIRED_LANGUAGES = ["TR", "KO", "FI", "EN"] + (["HU"] if HU_FILE else [])
PINNED_CONFIG = Path("configs/experiment_matrix.pinned.json")
MORPH_DATA = Path("data/processed/standardized_with_morphology.parquet")
FULL_PLAN = Path("artifacts/experiment_plan.pinned.json")
RESULT_ROOT = Path("outputs/encoder")

# Persistent caches outside Git artifacts.
os.environ["HF_HOME"] = "/content/drive/MyDrive/HBB_model_cache/huggingface"
os.environ["STANZA_RESOURCES_DIR"] = "/content/drive/MyDrive/HBB_model_cache/stanza"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["STANZA_RESOURCES_DIR"]).mkdir(parents=True, exist_ok=True)
print("Required languages:", REQUIRED_LANGUAGES)


## 3. Install the declared dependencies


In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])


## 4. Verify and record the runtime

The publication matrix is intentionally blocked on CPU. Package, Git, and GPU metadata are saved for the reproducibility appendix.


In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA is unavailable. Select an A100 GPU runtime before publication runs."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
run(["nvidia-smi"])

environment_dir = Path("artifacts/environment")
environment_dir.mkdir(parents=True, exist_ok=True)
(environment_dir / "git_commit.txt").write_text(
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True)
)
with (environment_dir / "pip_freeze.txt").open("w") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)
with (environment_dir / "nvidia_smi.txt").open("w") as handle:
    subprocess.run(["nvidia-smi"], stdout=handle, check=True)


## 5. Audit and standardize every dataset

This uses fixed hash manifests, pinned Git commits for BEEP!/TweetEval, and the corrected Finnish threshold. It never downloads an unverified Hungarian dataset by name.


In [ ]:
run([sys.executable, "scripts/audit_and_build_splits.py"])
run(["bash", "scripts/fetch_public_datasets.sh"])

prepare = [
    sys.executable,
    "scripts/prepare_standardized_datasets.py",
    "--korean-repo", "data/raw/korean-hate-speech",
    "--tweeteval-repo", "data/raw/tweeteval",
]
if HU_FILE is not None:
    HU_FILE = Path(HU_FILE)
    if not HU_FILE.exists():
        raise FileNotFoundError(HU_FILE)
    prepare += [
        "--hungarian", HU_FILE,
        "--hungarian-text-column", HU_TEXT_COLUMN,
        "--hungarian-label-column", HU_LABEL_COLUMN,
        "--hungarian-threshold", str(HU_THRESHOLD),
    ]
run(prepare)

run([sys.executable, "scripts/analyze_tokenization.py"])
run([sys.executable, "scripts/run_linear_diagnostics.py", "--output-dir", "artifacts/linear_diagnostics/l128"])
run([sys.executable, "scripts/run_linear_diagnostics.py", "--max-length", "0", "--output-dir", "artifacts/linear_diagnostics/full_text"])
run([sys.executable, "scripts/bootstrap_linear_diagnostics.py"])
run([sys.executable, "scripts/bootstrap_length_sensitivity.py"])
run([sys.executable, "scripts/make_dataset_tables.py"])
run([sys.executable, "scripts/make_figures.py"])
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"])
print("Verified CPU-stage analyses completed.")


## 6. Inspect the corrected evidence before training


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image

display(Markdown(Path("artifacts/datasets/PAPER_DATA_TABLES.md").read_text()))
display(Markdown(Path("artifacts/tokenization/REPORT.md").read_text()))
display(Markdown(Path("artifacts/linear_diagnostics/REPORT.md").read_text()))
display(Image(filename="manuscript/figures/figure_tokenization_truncation.png"))
display(Image(filename="manuscript/figures/figure_token_overlap.png"))
display(Image(filename="manuscript/figures/figure_linear_diagnostics.png"))


## 7. Build lemma, UD-feature, and randomized-placebo views

The randomized view preserves the language-specific feature-bundle inventory while breaking token–analysis correspondence. The generated QC sheet must be checked by qualified speakers/annotators; automatic Stanza output is not assumed to be gold morphology.


In [ ]:
if REBUILD_MORPHOLOGY or not MORPH_DATA.exists():
    run([
        sys.executable,
        "scripts/build_ud_morph_views.py",
        "data/processed/standardized_all.csv",
        MORPH_DATA,
        "--download",
        "--use-gpu",
        "--model-dir", os.environ["STANZA_RESOURCES_DIR"],
        "--batch-documents", "32",
    ])
else:
    print("Reusing:", MORPH_DATA)

qc_path = Path("artifacts/morphology_qc/morphology_qc_sample.csv")
if not qc_path.exists():
    run([
        sys.executable,
        "scripts/sample_morphology_qc.py",
        MORPH_DATA,
        "--output", qc_path,
        "--per-language", "100",
        "--seed", "42",
    ])
qc = pd.read_csv(qc_path)
display(qc.groupby("language").size().rename("rows"))
display(qc.head(3))
print("QC sheet:", qc_path.resolve())
if not MORPH_QC_APPROVED:
    print("MANUAL GATE: complete/review this sheet, then set MORPH_QC_APPROVED=True.")


## 8. Resolve immutable Hugging Face model commits

The mutable `main` tags in the template are resolved once. Reruns reuse the pinned file unless `REPIN_MODEL_REVISIONS=True`.


In [ ]:
if REPIN_MODEL_REVISIONS or not PINNED_CONFIG.exists():
    run([
        sys.executable,
        "scripts/pin_model_revisions.py",
        "--config", "configs/experiment_matrix.json",
        "--output", PINNED_CONFIG,
        "--manifest", "artifacts/model_revisions.json",
    ])
pinned = json.loads(PINNED_CONFIG.read_text())
for block in pinned["blocks"]:
    for model in block["models"]:
        print(block["name"], model["model_name"], model["model_revision"])
        assert len(model["model_revision"]) >= 40


## 9. Fail-fast preflight and inspect the exact 30-run plan


In [ ]:
run([
    sys.executable,
    "scripts/validate_confirmatory_setup.py",
    "--config", PINNED_CONFIG,
    "--output", "artifacts/confirmatory_preflight.json",
    "--require-cuda",
    "--require-languages", ",".join(REQUIRED_LANGUAGES),
])
run([
    sys.executable,
    "scripts/run_experiment_matrix.py",
    "--config", PINNED_CONFIG,
    "--plan", FULL_PLAN,
])
plan = pd.read_json(FULL_PLAN)
assert len(plan) == 30, f"Expected 30 runs, found {len(plan)}"
display(plan[["index", "block", "run_id", "metrics_path"]])


## 10. Execute/resume the GPU matrix

Completed `metrics.csv` files are skipped. The surface/model, length, and truly frozen blocks can run before manual morphology QC. The morphology block remains blocked until `MORPH_QC_APPROVED=True`. Do not change hyperparameters after examining test results.


In [ ]:
if RUN_GPU_MATRIX:
    surface_blocks = [
        "primary_surface_models",
        "finnish_length_sensitivity_xlmr_base",
        "true_frozen_encoder_ablation",
    ]
    for block in surface_blocks:
        run([
            sys.executable,
            "scripts/run_experiment_matrix.py",
            "--config", PINNED_CONFIG,
            "--plan", f"artifacts/plans/{block}.json",
            "--block", block,
            "--execute",
        ])

    if MORPH_QC_APPROVED:
        run([
            sys.executable,
            "scripts/run_experiment_matrix.py",
            "--config", PINNED_CONFIG,
            "--plan", "artifacts/plans/morphology_mechanism_xlmr_base.json",
            "--block", "morphology_mechanism_xlmr_base",
            "--execute",
        ])
    else:
        print("Morphology block skipped: manual QC has not been approved.")
else:
    print("RUN_GPU_MATRIX=False; no GPU training was started.")


## 11. Check completeness—never merge partial seeds


In [ ]:
plan_records = json.loads(FULL_PLAN.read_text())
completion = pd.DataFrame([
    {
        "run_id": item["run_id"],
        "block": item["block"],
        "complete": all(Path(path).exists() for path in item["required_outputs"]),
        "missing_outputs": ", ".join(
            path for path in item["required_outputs"] if not Path(path).exists()
        ),
    }
    for item in plan_records
])
display(completion.groupby("block")["complete"].agg(["sum", "count"]))
ALL_RUNS_COMPLETE = bool(completion["complete"].all())
print("All 30 runs complete:", ALL_RUNS_COMPLETE)
if not ALL_RUNS_COMPLETE:
    display(completion.loc[~completion["complete"]])


## 12. Aggregate, compare, and plot only complete results

The paired bootstrap resamples seeds and paired examples. The key morphology test is `morph_minus_randomized_morph`; a surface-language ranking alone is not treated as a morphology mechanism.


In [ ]:
if ALL_RUNS_COMPLETE:
    run([sys.executable, "scripts/aggregate_encoder_results.py", "--root", RESULT_ROOT])
    run([sys.executable, "scripts/bootstrap_encoder_comparisons.py", "--root", RESULT_ROOT])
    run([sys.executable, "scripts/make_encoder_figures.py"])
    display(Markdown(Path("artifacts/results/ENCODER_RESULTS.md").read_text()))
    display(Markdown(Path("artifacts/results/ENCODER_COMPARISONS.md").read_text()))
    display(Image(filename="artifacts/results/figures/figure_encoder_primary_models.png"))
    display(Image(filename="artifacts/results/figures/figure_encoder_morphology_views.png"))
    display(Image(filename="artifacts/results/figures/figure_encoder_finnish_length.png"))
else:
    print("Post-processing skipped: complete every seed/condition first.")


## 13. Create a blinded error-analysis set from the three-seed ensemble

Probabilities—not hard labels—are averaged across seeds 42/123/456. Up to 100 false positives and 100 false negatives per language are sampled. Availability is reported rather than silently duplicating scarce errors.


In [ ]:
if ALL_RUNS_COMPLETE:
    ensemble_path = Path("artifacts/error_annotation/xlmr_base_surface_ensemble_predictions.csv")
    run([
        sys.executable,
        "scripts/ensemble_encoder_predictions.py",
        "--root", RESULT_ROOT,
        "--model", "xlm-roberta-base",
        "--view", "surface",
        "--max-length", "128",
        "--output", ensemble_path,
    ])
    annotation_dir = Path("artifacts/error_annotation")
    run([
        sys.executable,
        "scripts/sample_linguistic_errors.py",
        ensemble_path,
        "--output-dir", annotation_dir,
        "--outcomes", "FP,FN",
        "--per-cell", str(ERRORS_PER_LANGUAGE_OUTCOME),
        "--seed", "42",
    ])
    template = annotation_dir / "annotation_blind.csv"
    for annotator in ("A", "B"):
        destination = annotation_dir / f"annotation_annotator_{annotator}.csv"
        if not destination.exists():
            shutil.copy2(template, destination)
    display(pd.read_csv(annotation_dir / "sampling_availability.csv"))
    display(Markdown(Path("docs/LINGUISTIC_ERROR_ANNOTATION_GUIDE_TR.md").read_text()))
    print("Give annotation_annotator_A.csv and annotation_annotator_B.csv to independent annotators.")
else:
    print("Error sample skipped until all model runs are complete.")


## 14. Agreement, adjudication, and linguistic summary

Run this cell after both annotators return their independently completed files. Keep `annotation_key.csv` hidden until coding is finished. Resolve disagreements in a separate `annotation_adjudicated.csv` file.


In [ ]:
annotation_dir = Path("artifacts/error_annotation")
annotator_a = annotation_dir / "annotation_annotator_A.csv"
annotator_b = annotation_dir / "annotation_annotator_B.csv"
adjudicated = annotation_dir / "annotation_adjudicated.csv"


def contains_codes(path):
    if not path.exists():
        return False
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    from scripts.sample_linguistic_errors import PHENOMENON_COLUMNS
    coding_columns = [column for column in PHENOMENON_COLUMNS if column in frame.columns]
    return bool(coding_columns) and frame[coding_columns].apply(lambda col: col.str.strip().ne("")).any().any()


if contains_codes(annotator_a) and contains_codes(annotator_b):
    run([
        sys.executable,
        "scripts/calculate_annotation_agreement.py",
        annotator_a,
        annotator_b,
        "--labels", "annotator_A,annotator_B",
    ])
    display(Markdown(Path("artifacts/error_annotation/agreement/AGREEMENT.md").read_text()))
else:
    print("Agreement not run: both completed annotation files are required.")

if adjudicated.exists():
    run([
        sys.executable,
        "scripts/summarize_error_annotations.py",
        adjudicated,
        annotation_dir / "annotation_key.csv",
    ])
    display(Markdown(Path("artifacts/error_annotation/summary/LINGUISTIC_ERROR_RESULTS.md").read_text()))
else:
    print("Create annotation_adjudicated.csv after resolving the listed disagreements.")


## 15. Final integrity check and paper-ready outputs

Only after all 30 runs, complete annotation, and internal review should the working-draft warning be removed. The notebook does not automatically write causal claims into the manuscript.


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"])
run([sys.executable, "-m", "compileall", "-q", "scripts", "tests"])

expected_outputs = [
    "artifacts/results/encoder_results.xlsx",
    "artifacts/results/encoder_paired_bootstrap.xlsx",
    "artifacts/results/ENCODER_RESULTS.md",
    "artifacts/results/ENCODER_COMPARISONS.md",
    "artifacts/model_revisions.json",
    "artifacts/confirmatory_preflight.json",
]
for output in expected_outputs:
    print("OK" if Path(output).exists() else "MISSING", output)
print("Do not report a condition whose file is missing or whose seed completeness check failed.")
